# LOAD REPO

In [ ]:
!git clone https://<GITHUB_TOKEN>@github.com/EkmalRey/FutsalCV.git

Cloning into 'FutsalCV'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 251 (delta 24), reused 49 (delta 17), pack-reused 193 (from 1)
Receiving objects: 100% (251/251), 264.49 MiB | 23.48 MiB/s, done.
Resolving deltas: 100% (106/106), done.
Updating files: 100% (26/26), done.


# INSTALL REQUIREMENTS

In [ ]:
# Install runtime dependencies (idempotent)
%pip install -r /content/FutsalCV/Resources/Notebooks/TRAINING/YOLO/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 20.2 MB/s eta 0:00:00


# IMPORTS

In [ ]:
from pathlib import Path
import json
import time
import math
import re
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Sequence
import sys

import numpy as np
import pandas as pd

try:
    import cv2
except ImportError:
    cv2 = None

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None

from tqdm import tqdm


# Point to helper folder (sibling of this notebook)
HELPER_DIR = Path("FutsalCV") / "Resources" / "Notebooks" / "TRAINING" / "YOLO" / "Helper"
sys.path.append(str(HELPER_DIR))

# Local helpers
from footballanalytix.config import ObjectDetectionConfig
from _5_evaluation_all import (
    prepare_dataset_and_gt,
    load_tracking_gt,
    summarize_gt,
    box_iou_matrix,
    average_precision,
    compute_detection_metrics,
    greedy_match,
    compute_tracking_metrics,
    clustering_purity,
    measure_fps,
    time_block,
    evaluate_tracking_on_video,
    collect_detection_samples_from_video,
    evaluate_clustering_on_video,
    run_full_evaluation,
    stitch_frames_to_video,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# CONFIG

In [ ]:
# Toggle for quick vs full run
DEBUG_QUICK = False  # full evaluation across the test split

PROJECT_ROOT = Path("/content/FutsalCV")
RESOURCES_DIR = PROJECT_ROOT / "Resources"
MODELS_DIR = RESOURCES_DIR / "Models"
DATA_ROOT = Path("./SN-GSR-2025")
TRACKING_OUT = Path("./eda_outputs/tracking_gt")

# Default model paths
PLAYERS_MODEL_PATH = MODELS_DIR / "players.pt"

# Test-only evaluation knobs
DOWNLOAD_TEST_ONLY = True  # stay on test split only
MAX_GAMES_DOWNLOAD = None  # evaluate all available test games
MAX_FRAMES_EVAL = None     # evaluate all frames per game

if not PLAYERS_MODEL_PATH.exists():
    print(f"⚠️ Players model missing at {PLAYERS_MODEL_PATH}; provide your checkpoint to run eval.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Resources dir: {RESOURCES_DIR}")
print(f"DEBUG_QUICK: {DEBUG_QUICK}")
print(f"Test-only download: {DOWNLOAD_TEST_ONLY}")

Project root: /content/FutsalCV
Resources dir: /content/FutsalCV/Resources
DEBUG_QUICK: False
Test-only download: True


# EVALUATION PROGRAM

## Dataset Download & GT Export

In [ ]:
# Download dataset (test split only) and export tracking GT JSONs
TRACKING_OUT.mkdir(parents=True, exist_ok=True)
DEBUG_TRAIN_ONLY = False
DEBUG_TEST_ONLY = DOWNLOAD_TEST_ONLY  # stay on test split
MAX_GAMES = MAX_GAMES_DOWNLOAD       # allow all test games
tracking_summary = prepare_dataset_and_gt(
    data_root=DATA_ROOT,
    debug_train_only=DEBUG_TRAIN_ONLY,
    debug_test_only=DEBUG_TEST_ONLY,
    tracking_out=TRACKING_OUT,
    max_games=MAX_GAMES,
    )
print(tracking_summary)

📥 Downloading SoccerNet GSR dataset...


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

test.zip:   0%|          | 0.00/8.85G [00:00<?, ?B/s]

📦 Extracting test.zip...
   ✅ Extracted to /content/SN-GSR-2025/test

✅ Dataset ready:
   📁 Test games : 49
   🧪 Debug mode: test-only download
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-116_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-117_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-118_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-119_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-120_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-121_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-122_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-123_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-124_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-125_tracking_gt.json
✓ Saved tracking GT -> eda_outputs/tracking_gt/test/SNGS-126_tracking_gt.jso

## GT Preparation

In [ ]:
# Discover all test GT JSONs
TEST_GT_PATH = TRACKING_OUT / "test"
TEST_GT_FILES = sorted(TEST_GT_PATH.glob("*_tracking_gt.json"))
if not TEST_GT_FILES:
    raise FileNotFoundError("No tracking GT JSONs found under test split. Run the export cell first.")

TEST_GT_INDEX = [
    {"game": fp.stem.replace("_tracking_gt", ""), "gt_path": fp} for fp in TEST_GT_FILES
]

print(f"Found {len(TEST_GT_INDEX)} test GT files")
print([item["gt_path"].name for item in TEST_GT_INDEX])

# Register test frame folders
TEST_GAME_DIR = DATA_ROOT / "test"
_test_games = sorted(TEST_GAME_DIR.glob("SNGS-*"))
if not _test_games:
    raise FileNotFoundError("No test games found. Ensure dataset is downloaded.")

TEST_EVAL_JOBS = []
for game_dir in _test_games:
    frames_dir = game_dir / "img1"
    frame_files = sorted(frames_dir.glob("*.jpg"))
    if not frame_files:
        print(f"⚠️ No frames found in {frames_dir}, skipping.")
        continue
    frame_count = len(frame_files)
    effective_frames = frame_count if MAX_FRAMES_EVAL is None else min(frame_count, int(MAX_FRAMES_EVAL))
    gt_candidate = TEST_GT_PATH / f"{game_dir.name}_tracking_gt.json"
    if not gt_candidate.exists():
        print(f"⚠️ GT missing for {game_dir.name}, skipping this game.")
        continue
    TEST_EVAL_JOBS.append({
        "game": game_dir.name,
        "frames_path": frames_dir,
        "gt_path": gt_candidate,
        "frame_count": effective_frames,
    })

print(f"Prepared {len(TEST_EVAL_JOBS)} test games for evaluation")
print(TEST_EVAL_JOBS)

Found 49 test GT files
['SNGS-116_tracking_gt.json', 'SNGS-117_tracking_gt.json', 'SNGS-118_tracking_gt.json', 'SNGS-119_tracking_gt.json', 'SNGS-120_tracking_gt.json', 'SNGS-121_tracking_gt.json', 'SNGS-122_tracking_gt.json', 'SNGS-123_tracking_gt.json', 'SNGS-124_tracking_gt.json', 'SNGS-125_tracking_gt.json', 'SNGS-126_tracking_gt.json', 'SNGS-127_tracking_gt.json', 'SNGS-128_tracking_gt.json', 'SNGS-129_tracking_gt.json', 'SNGS-130_tracking_gt.json', 'SNGS-131_tracking_gt.json', 'SNGS-132_tracking_gt.json', 'SNGS-133_tracking_gt.json', 'SNGS-134_tracking_gt.json', 'SNGS-135_tracking_gt.json', 'SNGS-136_tracking_gt.json', 'SNGS-137_tracking_gt.json', 'SNGS-138_tracking_gt.json', 'SNGS-139_tracking_gt.json', 'SNGS-140_tracking_gt.json', 'SNGS-141_tracking_gt.json', 'SNGS-142_tracking_gt.json', 'SNGS-143_tracking_gt.json', 'SNGS-144_tracking_gt.json', 'SNGS-145_tracking_gt.json', 'SNGS-146_tracking_gt.json', 'SNGS-147_tracking_gt.json', 'SNGS-148_tracking_gt.json', 'SNGS-149_tracking_

## Full Evaluation

In [ ]:
# Comprehensive evaluation run across all prepared test games (homography disabled)
import _5_evaluation_all as eval_helpers

# Disable homography evaluation inside the helper to avoid keypoint/pitch requirements
eval_helpers.evaluate_homography_on_video = lambda *args, **kwargs: {"mean_reprojection_error": 0.0, "frames_used": 0}

if not TEST_EVAL_JOBS:
    raise RuntimeError("No prepared test games to evaluate. Register frame folders first.")

# Pretty printing helpers
def format_job(job: dict) -> str:
    return (
        f"game: {job['game']} | frames: {job['frame_count']} | "
        f"frames_path: {job['frames_path']} | gt: {job['gt_path']}"
    )

print("=== Evaluation plan ===")
for job in TEST_EVAL_JOBS:
    print(format_job(job))
print("=======================\n")

full_eval_results = []
for job in TEST_EVAL_JOBS:
    if not job["frames_path"].exists():
        raise FileNotFoundError(f"Frames not found at {job['frames_path']} for {job['game']}")
    if not job["gt_path"].exists():
        raise FileNotFoundError(f"GT not found at {job['gt_path']} for {job['game']}")

    start_time = time.time()
    with time_block(f"full_evaluation_{job['game']}"):
        results = run_full_evaluation(
            job["frames_path"],
            job["gt_path"],
            PLAYERS_MODEL_PATH,
            max_frames=job["frame_count"],
            field_model_path=None,
            pitch_json=None,
            verbose=False,
        )
    elapsed = time.time() - start_time

    det = results.get("detection", {}) or {}
    det_map = det.get("mAP", 0.0)
    det_mean_iou = det.get("mean_iou", 0.0)
    det_per_class_iou = {k: v.get("mean_iou", 0.0) for k, v in (det.get("per_class", {}) or {}).items()}

    full_eval_results.append({
        "game": job["game"],
        "frames": job["frame_count"],
        "elapsed_s": round(elapsed, 2),
        "det_mAP": det_map,
        "det_mean_iou": det_mean_iou,
        "det_per_class_iou": det_per_class_iou,
        **results,
    })

if full_eval_results:
    # Use a compact table view for readability
    df = pd.DataFrame(full_eval_results)
    preferred = ["game", "frames", "elapsed_s", "det_mAP", "det_mean_iou"]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    print("=== Evaluation results ===")
    print(df[cols].to_string(index=False))
    print("=========================")

    # Persist raw JSON payload for downstream use
    out_path = Path("full_eval_result.json")
    out_path.write_text(json.dumps(full_eval_results, indent=2))
    print(f"Saved full evaluation JSON to {out_path.resolve()}")
else:
    print("No evaluation results produced.")


=== Evaluation plan ===
game: SNGS-116 | frames: 750 | frames_path: SN-GSR-2025/test/SNGS-116/img1 | gt: eda_outputs/tracking_gt/test/SNGS-116_tracking_gt.json
game: SNGS-117 | frames: 750 | frames_path: SN-GSR-2025/test/SNGS-117/img1 | gt: eda_outputs/tracking_gt/test/SNGS-117_tracking_gt.json
game: SNGS-118 | frames: 750 | frames_path: SN-GSR-2025/test/SNGS-118/img1 | gt: eda_outputs/tracking_gt/test/SNGS-118_tracking_gt.json
game: SNGS-119 | frames: 750 | frames_path: SN-GSR-2025/test/SNGS-119/img1 | gt: eda_outputs/tracking_gt/test/SNGS-119_tracking_gt.json
game: SNGS-120 | frames: 750 | frames_path: SN-GSR-2025/test/SNGS-120/img1 | gt: eda_outputs/tracking_gt/test/SNGS-120_tracking_gt.json
game: SNGS-121 | frames: 750 | frames_path: SN-GSR-2025/test/SNGS-121/img1 | gt: eda_outputs/tracking_gt/test/SNGS-121_tracking_gt.json
game: SNGS-122 | frames: 750 | frames_path: SN-GSR-2025/test/SNGS-122/img1 | gt: eda_outputs/tracking_gt/test/SNGS-122_tracking_gt.json
game: SNGS-123 | frames: